In [ ]:
!pip install -U gliner tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.4/170.4 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 130.7 MB/s eta 0:00:00


In [ ]:
import json
import time
import sys
from pathlib import Path

from tqdm import tqdm
from gliner import GLiNER

# ─────────────────────────────────────────────
# 1. LOAD GLINER MODEL
# ─────────────────────────────────────────────

print("Loading GLiNER model...")
model = GLiNER.from_pretrained("urchade/gliner_multi-v2.1")
print("✅ Model loaded")

# ─────────────────────────────────────────────
# 2. LOAD DATA
# ─────────────────────────────────────────────

def load_sentences(filepath: str):

    path = Path(filepath)

    if not path.exists():
        print(f"❌ File not found: {filepath}")
        sys.exit(1)

    if path.suffix == ".json":

        with open(path, encoding="utf-8") as f:
            data = json.load(f)

        if len(data) == 0:
            return []

        if isinstance(data[0], dict):
            sentences = [
                d["text"]
                for d in data
                if d.get("text", "").strip()
            ]
        else:
            sentences = [
                s.strip()
                for s in data
                if s.strip()
            ]

    elif path.suffix == ".txt":

        with open(path, encoding="utf-8") as f:
            sentences = [
                line.strip()
                for line in f
                if line.strip()
            ]

    else:
        print("❌ Unsupported format (.txt or .json)")
        sys.exit(1)

    return sentences


# ─────────────────────────────────────────────
# 3. GLINER NER ANNOTATION
# ─────────────────────────────────────────────

def annotate_sentence(sentence: str):

    tokens = sentence.split()

    if not tokens:
        return ""

    labels = ["O"] * len(tokens)

    token_spans = []
    current_pos = 0

    for token in tokens:

        start = sentence.find(token, current_pos)
        end = start + len(token)

        token_spans.append((start, end))
        current_pos = end

    try:

        entities = model.predict_entities(
            sentence,
            ["person", "location", "organization"],
            threshold=0.4
        )

        label_map = {
            "person": "PER",
            "location": "LOC",
            "organization": "ORG"
        }

        for ent in entities:

            if ent["label"] not in label_map:
                continue

            entity_type = label_map[ent["label"]]

            start_ent = ent["start"]
            end_ent = ent["end"]

            first_token = True

            for i, (tok_start, tok_end) in enumerate(token_spans):

                overlap = (
                    tok_start < end_ent
                    and tok_end > start_ent
                )

                if overlap:

                    if first_token:
                        labels[i] = f"B-{entity_type}"
                        first_token = False
                    else:
                        labels[i] = f"I-{entity_type}"

    except Exception as e:
        print(f"⚠️ Annotation error: {e}")

    return "\n".join(
        f"{token}\t{label}"
        for token, label in zip(tokens, labels)
    )


def annotate_batch(sentences):

    results = []

    for sentence in sentences:
        results.append(
            annotate_sentence(sentence)
        )

    return "\n\n".join(results)


# ─────────────────────────────────────────────
# 4. SAVE OUTPUT
# ─────────────────────────────────────────────

def save_conll(annotations, output_path):

    with open(output_path, "w", encoding="utf-8") as f:
        f.write("\n".join(annotations))

    print(f"✅ Saved to {output_path}")


# ─────────────────────────────────────────────
# 5. MAIN PIPELINE
# ─────────────────────────────────────────────

def annotate_file(
    input_path: str,
    output_path: str = None,
    batch_size: int = 10
):

    if output_path is None:
        output_path = str(
            Path(input_path).with_suffix(".conll")
        )

    print(f"📂 Loading: {input_path}")

    sentences = load_sentences(input_path)

    total = len(sentences)

    print(f"📊 Total sentences: {total}")
    print(f"🔄 Batch size: {batch_size}")

    all_annotations = []
    failed_batches = []

    batch_indices = range(0, total, batch_size)

    for i in tqdm(
        batch_indices,
        desc="Annotating"
    ):

        batch = sentences[i:i + batch_size]
        batch_num = (i // batch_size) + 1

        try:

            result = annotate_batch(batch)
            all_annotations.append(result)

        except Exception as e:

            print(
                f"❌ Batch {batch_num} failed: {e}"
            )

            failed_batches.append(batch_num)

            for sentence in batch:

                tokens = sentence.split()

                fallback = "\n".join(
                    f"{token}\tO"
                    for token in tokens
                )

                all_annotations.append(fallback)

        time.sleep(0.1)

    save_conll(
        all_annotations,
        output_path
    )

    print("\n" + "=" * 50)
    print("SUMMARY")
    print("=" * 50)
    print(f"Total sentences : {total}")
    print(f"Failed batches  : {failed_batches}")
    print(f"Output file     : {output_path}")
    print("=" * 50)


# ─────────────────────────────────────────────
# 6. RUN
# ─────────────────────────────────────────────

if __name__ == "__main__":

    INPUT_FILE = "/content/drive/MyDrive/NER/final_dataset.txt"
    OUTPUT_FILE = "/content/drive/MyDrive/NER/train.conll"

    annotate_file(
        input_path=INPUT_FILE,
        output_path=OUTPUT_FILE,
        batch_size=10
    )

Loading GLiNER model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:189: UserWarning: The `resume_download` argument is deprecated and ignored in `snapshot_download`. Downloads always resume whenever possible.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

✅ Model loaded
📂 Loading: /content/drive/MyDrive/NER/final_dataset.txt
📊 Total sentences: 68774
🔄 Batch size: 10


Annotating:  67%|██████▋   | 4614/6878 [34:11<15:53,  2.37it/s]/usr/local/lib/python3.12/dist-packages/gliner/data_processing/processor.py:395: UserWarning: Sentence of length 1400 has been truncated to 384
  batch = [self.preprocess_example(b["tokenized_text"], b[key], class_to_ids) for b in batch_list]
Annotating:  67%|██████▋   | 4616/6878 [34:12<19:25,  1.94it/s]/usr/local/lib/python3.12/dist-packages/gliner/data_processing/processor.py:395: UserWarning: Sentence of length 1005 has been truncated to 384
  batch = [self.preprocess_example(b["tokenized_text"], b[key], class_to_ids) for b in batch_list]
Annotating:  68%|██████▊   | 4647/6878 [34:29<20:59,  1.77it/s]/usr/local/lib/python3.12/dist-packages/gliner/data_processing/processor.py:395: UserWarning: Sentence of length 1001 has been truncated to 384
  batch = [self.preprocess_example(b["tokenized_text"], b[key], class_to_ids) for b in batch_list]
Annotating:  95%|█████████▌| 6538/6878 [47:32<02:32,  2.23it/s]/usr/local/lib/pyth

✅ Saved to /content/drive/MyDrive/NER/train.conll

SUMMARY
Total sentences : 68774
Failed batches  : []
Output file     : /content/drive/MyDrive/NER/train.conll
